# 03 - CrewAI：角色驱动的 Agent 协作框架

## 学习目标

- 理解 CrewAI 的角色驱动设计理念
- 掌握 Agent、Task、Crew 的核心概念
- 学习构建角色化多 Agent 团队
- 实现一个完整的 CrewAI 项目

---

## 1. CrewAI 概述

### 1.1 什么是 CrewAI？

**CrewAI** 是一个用于编排角色化 Agent 协作的框架。与 AutoGen 的对话驱动不同，CrewAI 强调：

- **角色定义**：每个 Agent 有明确的角色、目标和背景故事
- **任务分配**：明确的任务分解和分配机制
- **流程控制**：支持顺序、并行、分层等执行流程
- **工具集成**：灵活的工具使用能力

### 1.2 CrewAI 的核心概念

| 概念 | 描述 | 类比 |
|------|------|------|
| **Agent** | 有角色的智能体 | 团队成员 |
| **Task** | 具体任务定义 | 工作项 |
| **Crew** | Agent 团队 + 任务集合 | 项目组 |
| **Process** | 执行流程 | 工作流 |
| **Tool** | Agent 可用的工具 | 技能/设备 |

### 1.3 CrewAI 架构图

```
┌─────────────────────────────────────────────────────────────┐
│                     CrewAI 架构                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌─────────────────────────────────────────────────────┐   │
│  │                      Crew                           │   │
│  │              (团队 + 任务 + 流程)                    │   │
│  └─────────────────────────────────────────────────────┘   │
│                          │                                  │
│          ┌───────────────┼───────────────┐                 │
│          │               │               │                 │
│          ▼               ▼               ▼                 │
│  ┌───────────┐   ┌───────────┐   ┌───────────┐            │
│  │  Agent 1  │   │  Agent 2  │   │  Agent 3  │            │
│  │ 研究员    │   │  写手     │   │  编辑     │            │
│  └───────────┘   └───────────┘   └───────────┘            │
│       │               │               │                    │
│       └───────────────┼───────────────┘                    │
│                       │                                     │
│                       ▼                                     │
│              ┌───────────────┐                             │
│              │    Tasks      │                             │
│              │  1 → 2 → 3    │                             │
│              └───────────────┘                             │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

---

## 2. 核心概念详解

### 2.1 Agent（智能体）

Agent 是 CrewAI 中的核心角色，每个 Agent 有三个关键属性：

```python
from crewai import Agent

researcher = Agent(
    role='高级研究员',
    goal='深入研究主题并收集关键信息',
    backstory='你是一位经验丰富的研究员，擅长深度分析和信息整合',
    verbose=True,
    allow_delegation=False,
    tools=[search_tool, web_scraper]
)
```

**属性说明**：

| 属性 | 说明 |
|------|------|
| `role` | Agent 的角色名称，如"研究员"、"写手" |
| `goal` | Agent 的核心目标 |
| `backstory` | 背景故事，影响 Agent 的行为风格 |
| `verbose` | 是否输出详细日志 |
| `allow_delegation` | 是否允许将任务委托给其他 Agent |
| `tools` | Agent 可用的工具列表 |

### 2.2 Task（任务）

Task 定义了 Agent 需要完成的具体工作：

```python
from crewai import Task

research_task = Task(
    description='研究 AI Agent 的最新发展趋势',
    expected_output='一份包含5个关键趋势的详细报告',
    agent=researcher,
    context=[previous_task]  # 可选：依赖其他任务的输出
)
```

**属性说明**：

| 属性 | 说明 |
|------|------|
| `description` | 任务描述 |
| `expected_output` | 期望的输出格式和内容 |
| `agent` | 执行任务的 Agent |
| `context` | 依赖的其他任务（用于任务链） |

### 2.3 Crew（团队）

Crew 将 Agent 和 Task 组合在一起，定义执行流程：

```python
from crewai import Crew, Process

crew = Crew(
    agents=[researcher, writer, editor],
    tasks=[research_task, writing_task, editing_task],
    process=Process.sequential,  # 或 Process.parallel, Process.hierarchical
    verbose=True
)

result = crew.kickoff()
```

**Process 类型**：

| 类型 | 说明 | 适用场景 |
|------|------|----------|
| `sequential` | 按顺序执行任务 | 有依赖关系的任务链 |
| `parallel` | 并行执行任务 | 独立任务 |
| `hierarchical` | 分层执行，有管理者 | 复杂项目 |

---

## 3. 动手实现：Mock CrewAI

下面我们实现一个简化版的 CrewAI，帮助理解其工作原理：



---
## 0. 模型准备：加载 Qwen2.5-7B-Instruct

本 Notebook 使用 **ModelScope** 加载本地 Qwen 模型，替代在线 API 调用。

- **推荐模型**：`Qwen/Qwen2.5-7B-Instruct`（约 15GB 显存）
- **低显存备选**：`Qwen/Qwen2.5-3B-Instruct`（约 6GB 显存）
- 自动检测 GPU / CPU，优先使用 GPU 加速

> 如果没有安装 modelscope 或显存不足，可以使用下方代码中的 **MockLLM 备选方案**。



In [ ]:
# ============================================================
# 安装依赖（首次运行时取消注释）
# ============================================================
# !pip install modelscope torch transformers -q

import torch

# ============================================================
# GPU / CPU 自动检测
# ============================================================
if torch.cuda.is_available():
    DEVICE = "cuda"
    GPU_NAME = torch.cuda.get_device_name(0)
    print(f"检测到 GPU: {GPU_NAME}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("检测到 Apple Silicon GPU (MPS)")
else:
    DEVICE = "cpu"
    print("未检测到 GPU，将使用 CPU（速度较慢）")

# ============================================================
# QwenLLM 封装类
# ============================================================
from modelscope import AutoModelForCausalLM, AutoTokenizer

class QwenLLM:
    """基于 ModelScope 的 Qwen 模型封装"""

    def __init__(self, model_name="Qwen/Qwen2.5-7B-Instruct", device=None):
        # 自动检测设备
        if device is None:
            device = "cuda" if torch.cuda.is_available() else (
                "mps" if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
                else "cpu"
            )
        self.device = device
        print(f"正在加载模型 {model_name}，设备: {device} ...")
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype="auto", device_map="auto"
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.messages = []
        print("模型加载完成！")

    def chat(self, user_message, system_prompt=None, max_new_tokens=512, temperature=0.7):
        """对话接口"""
        if system_prompt:
            messages = [{"role": "system", "content": system_prompt}]
        else:
            messages = []
        messages.extend(self.messages)
        messages.append({"role": "user", "content": user_message})

        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=True
        )
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        self.messages.append({"role": "user", "content": user_message})
        self.messages.append({"role": "assistant", "content": response})
        return response

    def reset(self):
        """清空对话历史"""
        self.messages = []

# ============================================================
# 初始化模型
# ============================================================
# 低显存环境可切换为: Qwen/Qwen2.5-3B-Instruct
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

try:
    llm = QwenLLM(model_name=MODEL_NAME)
    USE_REAL_MODEL = True
    print("\n使用真实 Qwen 模型")
except Exception as e:
    print(f"\n模型加载失败: {e}")
    print("将使用 MockLLM 备选方案")
    USE_REAL_MODEL = False

print(f"\n当前设备: {DEVICE}")
print(f"使用真实模型: {USE_REAL_MODEL}")



In [ ]:
# ============================================================
# 无模型时的备选方案：MockLLM
# （仅在上方模型加载失败时使用）
# ============================================================

if not USE_REAL_MODEL:
    class MockLLM:
        """模拟 LLM，用于无模型环境下的教学演示"""

        def __init__(self, model_name="mock-qwen"):
            self.model_name = model_name
            self.messages = []

        def chat(self, user_message, system_prompt=None, max_new_tokens=512, temperature=0.7):
            """模拟对话回复"""
            if '你好' in user_message:
                response = "你好！我是 AI 助手，有什么可以帮助你的吗？"
            elif '天气' in user_message:
                response = "我无法获取实时天气信息，建议您查看天气应用。"
            elif 'LangChain' in user_message:
                response = "LangChain 是一个用于开发 LLM 应用的框架，提供了 Chains、Prompts、Memory 等核心组件。"
            elif 'Agent' in user_message:
                response = "Agent 是一种能够自主决策并调用工具的智能体，通常使用 ReAct 模式工作。"
            else:
                response = f"我收到了您的消息：'{user_message[:50]}'。这是一个模拟回复。"

            self.messages.append({"role": "user", "content": user_message})
            self.messages.append({"role": "assistant", "content": response})
            return response

        def reset(self):
            self.messages = []

    llm = MockLLM(model_name="mock-qwen")
    print("已启用 MockLLM 备选方案")

# 测试模型调用
response = llm.chat("你好，请介绍一下自己")
print(f"模型回复: {response}")



In [ ]:
# 模拟 CrewAI Agent（使用 QwenLLM 执行任务）

# --- 无模型时的备选方案：原始 MockAgent（已注释）---
# class MockAgent:
#     def _simulate_work(self, task_description, context):
#         if "研究" in task_description:
#             return f"[{self.role}] 研究报告：..."
#         ...

class MockAgent:
    """模拟 CrewAI Agent（使用 QwenLLM 执行任务）"""

    def __init__(self, role, goal, backstory, verbose=False, allow_delegation=True, tools=None):
        self.role = role
        self.goal = goal
        self.backstory = backstory
        self.verbose = verbose
        self.allow_delegation = allow_delegation
        self.tools = tools or []

    def execute(self, task_description, context=None):
        """执行任务（使用 QwenLLM）"""
        if self.verbose:
            print(f"\n[{self.role}] 开始执行任务")
            print(f"   目标: {self.goal}")
            print(f"   任务: {task_description}")
            if context:
                print(f"   上下文: {context[:100]}...")

        # 使用 QwenLLM 生成任务结果
        result = self._generate_with_llm(task_description, context)

        if self.verbose:
            print(f"[{self.role}] 任务完成")

        return result

    def _generate_with_llm(self, task_description, context):
        """使用 QwenLLM 生成任务结果"""
        try:
            # 构建角色化提示词
            prompt = f"你是 {self.role}。{self.backstory}\n\n"
            prompt += f"你的目标：{self.goal}\n\n"
            prompt += f"当前任务：{task_description}\n"
            if context:
                prompt += f"\n参考上下文：\n{context}\n"
            prompt += f"\n请以 {self.role} 的身份完成上述任务，给出专业的输出。"

            result = llm.chat(prompt, system_prompt=f"你是{self.role}。{self.backstory}", max_new_tokens=512)
            return f"[{self.role}] {result}"
        except Exception as e:
            # 备选：基于关键词的简单输出
            return self._simulate_work_fallback(task_description, context)

    def _simulate_work_fallback(self, task_description, context):
        """备选方案：无模型时的简单模拟"""
        if "研究" in task_description or "调查" in task_description:
            return f"[{self.role}] 研究报告：\n- 发现1：相关领域的最新进展\n- 发现2：关键技术突破\n- 发现3：未来发展趋势"
        elif "写" in task_description or "撰写" in task_description:
            if context:
                return f"[{self.role}] 基于研究成果撰写的文章：\n标题：深入分析\n\n{context[:200]}...\n\n[完整文章内容]"
            return f"[{self.role}] 撰写的文章：\n标题：新主题探索\n\n[文章内容...]"
        elif "编辑" in task_description or "审校" in task_description or "优化" in task_description:
            return f"[{self.role}] 编辑后的版本：\n- 修正了语法错误\n- 优化了段落结构\n- 提升了可读性\n\n{context[:200] if context else ''}..."
        else:
            return f"[{self.role}] 任务完成：{task_description}"

    def __repr__(self):
        return f"MockAgent(role='{self.role}', goal='{self.goal[:20]}...')"

# 测试 Agent
researcher = MockAgent(
    role='高级研究员',
    goal='深入研究主题并收集关键信息',
    backstory='你是一位经验丰富的研究员',
    verbose=True
)

result = researcher.execute("研究 AI Agent 的最新发展趋势")
print("\n" + "="*50)
print(result)



In [ ]:
class MockTask:
    """模拟 CrewAI Task"""

    def __init__(self, description, expected_output, agent, context=None):
        self.description = description
        self.expected_output = expected_output
        self.agent = agent
        self.context = context or []
        self.output = None

    def execute(self):
        """执行任务"""
        # 收集上下文
        context_str = "\n\n".join([ctx.output for ctx in self.context if ctx.output])

        self.output = self.agent.execute(
            self.description,
            context=context_str if context_str else None
        )
        return self.output

    def __repr__(self):
        return f"MockTask('{self.description[:30]}...')"

# 创建任务
research_task = MockTask(
    description='研究 AI Agent 的最新发展趋势',
    expected_output='一份包含5个关键趋势的详细报告',
    agent=researcher
)

result = research_task.execute()
print(result)



In [ ]:
from enum import Enum

class MockProcess(Enum):
    """模拟 CrewAI Process 类型"""
    sequential = "sequential"
    parallel = "parallel"
    hierarchical = "hierarchical"

class MockCrew:
    """模拟 CrewAI Crew"""

    def __init__(self, agents, tasks, process=MockProcess.sequential, verbose=True):
        self.agents = agents
        self.tasks = tasks
        self.process = process
        self.verbose = verbose

    def kickoff(self):
        """启动 Crew 执行任务"""
        if self.verbose:
            print("\n" + "="*60)
            print(f"Crew 启动 - 流程类型: {self.process.value}")
            print(f"   Agent 数量: {len(self.agents)}")
            print(f"   任务数量: {len(self.tasks)}")
            print("="*60)

        if self.process == MockProcess.sequential:
            return self._run_sequential()
        elif self.process == MockProcess.parallel:
            return self._run_parallel()
        elif self.process == MockProcess.hierarchical:
            return self._run_hierarchical()

    def _run_sequential(self):
        """顺序执行"""
        if self.verbose:
            print("\n顺序执行模式")

        results = []
        for i, task in enumerate(self.tasks, 1):
            if self.verbose:
                print(f"\n--- 任务 {i}/{len(self.tasks)} ---")
            result = task.execute()
            results.append(result)

        return "\n\n".join(results)

    def _run_parallel(self):
        """并行执行（模拟）"""
        if self.verbose:
            print("\n并行执行模式")

        results = []
        for i, task in enumerate(self.tasks, 1):
            if self.verbose:
                print(f"\n--- 任务 {i}/{len(self.tasks)} (并行) ---")
            result = task.execute()
            results.append(result)

        return "\n\n".join(results)

    def _run_hierarchical(self):
        """分层执行（模拟）"""
        if self.verbose:
            print("\n分层执行模式")
            print("   管理者: 项目经理")

        # 模拟管理者分配任务
        return self._run_sequential()

# 创建完整的 Crew
researcher = MockAgent(
    role='高级研究员',
    goal='深入研究主题并收集关键信息',
    backstory='经验丰富的研究员',
    verbose=True
)

writer = MockAgent(
    role='技术写手',
    goal='将研究成果转化为高质量文章',
    backstory='资深技术写作者',
    verbose=True
)

editor = MockAgent(
    role='内容编辑',
    goal='确保内容质量和准确性',
    backstory='专业编辑',
    verbose=True
)

# 创建任务（带依赖关系）
task1 = MockTask(
    description='研究 AI Agent 的最新发展趋势',
    expected_output='研究报告',
    agent=researcher
)

task2 = MockTask(
    description='撰写关于 AI Agent 趋势的文章',
    expected_output='技术文章',
    agent=writer,
    context=[task1]
)

task3 = MockTask(
    description='编辑和校对文章',
    expected_output='最终版本',
    agent=editor,
    context=[task2]
)

# 创建 Crew 并执行
crew = MockCrew(
    agents=[researcher, writer, editor],
    tasks=[task1, task2, task3],
    process=MockProcess.sequential,
    verbose=True
)

final_result = crew.kickoff()
print("\n" + "="*60)
print("最终结果:")
print("="*60)
print(final_result[:500] + "...")



---

## 4. 实战案例：内容创作团队

下面我们模拟一个完整的内容创作流程：



In [ ]:
# 创建专业团队
topic = "AI Agent 在 2025 年的发展趋势"

# 研究员
researcher = MockAgent(
    role='行业分析师',
    goal='收集和分析行业数据',
    backstory='你是一位在 AI 领域工作10年的分析师',
    verbose=True
)

# 写手
writer = MockAgent(
    role='技术博客作者',
    goal='创作引人入胜的技术内容',
    backstory='你是一位知名的技术博客作者，擅长将复杂概念简单化',
    verbose=True
)

# SEO 专家
seo_expert = MockAgent(
    role='SEO 专家',
    goal='优化内容的搜索引擎可见性',
    backstory='你是一位资深的 SEO 专家，了解搜索引擎算法',
    verbose=True
)

# 创建任务
research_task = MockTask(
    description=f'研究 {topic} 的相关数据和趋势',
    expected_output='数据分析报告',
    agent=researcher
)

writing_task = MockTask(
    description=f'撰写关于 {topic} 的博客文章',
    expected_output='2000字的技术博客文章',
    agent=writer,
    context=[research_task]
)

seo_task = MockTask(
    description='优化文章的 SEO 表现',
    expected_output='SEO 优化后的文章',
    agent=seo_expert,
    context=[writing_task]
)

# 执行
content_crew = MockCrew(
    agents=[researcher, writer, seo_expert],
    tasks=[research_task, writing_task, seo_task],
    process=MockProcess.sequential,
    verbose=True
)

content = content_crew.kickoff()
print("\n" + "="*60)
print("最终内容:")
print("="*60)
print(content[:800] + "...")



---

## 5. CrewAI vs AutoGen vs LangChain

| 特性 | CrewAI | AutoGen | LangChain |
|------|--------|---------|-----------|
| **核心范式** | 角色驱动 | 对话驱动 | 组件组合 |
| **Agent 定义** | 角色+目标+背景 | 系统消息 | 提示词+工具 |
| **协作方式** | 任务分配 | 自然语言对话 | Chain/Graph |
| **流程控制** | Sequential/Parallel/Hierarchical | 对话终止条件 | StateGraph |
| **代码执行** | 通过工具 | 内置支持 | 通过工具 |
| **适用场景** | 内容创作、研究 | 编程、调试 | 通用工作流 |
| **学习曲线** | 中等 | 中等 | 较低 |

---

## 6. 小结

### 核心要点

1. **CrewAI** 是角色驱动的 Agent 协作框架
2. **核心概念**：Agent（角色）、Task（任务）、Crew（团队）
3. **角色定义**：每个 Agent 有 role、goal、backstory
4. **流程类型**：Sequential、Parallel、Hierarchical
5. **任务依赖**：通过 context 参数传递任务上下文

### 下一步

- [04_llamindex_rag_agent.ipynb](04_llamindex_rag_agent.ipynb) - 学习 LlamaIndex RAG Agent
- [05_google_adk.ipynb](05_google_adk.ipynb) - 探索 Google ADK

---

## 参考资源

- [CrewAI 官方文档](https://docs.crewai.com/)
- [CrewAI GitHub](https://github.com/joaomdmoura/crewai)
- [CrewAI 示例](https://github.com/joaomdmoura/crewai-examples)

